# Генерация таблицы данных для M2 пайлайна

In [1]:
import sys
from dotenv import load_dotenv

sys.path.append('./../')
load_dotenv()

True

In [2]:
from typing import Any, List, TypedDict
import json
from langchain_openai import OpenAIEmbeddings
from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize
import numpy as np

from generators.m2_generator import MDGenerationFlow, MDGenerationState, MetadataStructure

In [3]:
MODEL = 'gpt-oss-120b/latest' #'gpt-5.4-mini'
MODEL_PROVIDER = 'yandex'#'openai'
TEMPERATURE = 0.5

def read_prompts(pr_path:str) -> str:
    with open(pr_path, 'rt') as fio:
        res = fio.read()
    return res 

In [4]:
import time
from dataclasses import dataclass
from typing import Optional
from langchain_core.callbacks import BaseCallbackHandler


class TokenCounter(BaseCallbackHandler):
    def __init__(self):
        self.prompt_tokens     = 0
        self.completion_tokens = 0
        self.total_tokens      = 0
        self.n_calls           = 0

    def on_llm_end(self, response, **kwargs):
        usage = None

        # LangChain wraps different providers differently
        if hasattr(response, "llm_output") and response.llm_output:
            usage = response.llm_output.get("token_usage") or \
                    response.llm_output.get("usage")

        # OpenAI via langchain_openai exposes it here
        if usage is None and response.generations:
            for gen_list in response.generations:
                for gen in gen_list:
                    info = getattr(gen, "generation_info", None) or {}
                    usage = info.get("usage") or info.get("token_usage")
                    if usage:
                        break

        if usage:
            self.prompt_tokens     += usage.get("prompt_tokens",     0)
            self.completion_tokens += usage.get("completion_tokens", 0)
            self.total_tokens      += usage.get("total_tokens",      0)
        self.n_calls += 1

In [5]:
class QueryDescr(TypedDict):
    id: str
    topic: str
    field_of_science: str
    llm_query: str

def read_tasks(tasks_file: str) -> List[str]:
    with open(tasks_file, 'rt') as fio:
        d = json.load(fio)
        queries_list = []
        for el in d:
            queries_list.append(QueryDescr(el))
    return queries_list

def generate_meta(md_gen_wf: MDGenerationFlow, query: QueryDescr) -> MetadataStructure:
    counter = TokenCounter()

    md_gen_state = MDGenerationState()
    wf = md_gen_wf.workflow
    
    t0 = time.perf_counter()
    md_gen_state['users_task'] = query['llm_query']
    res_state = wf.invoke(
        md_gen_state,
        config={"callbacks": [counter]},
    )
    elapsed = time.perf_counter() - t0

    tokens_dict = {
        'prompt': counter.prompt_tokens,
        'completion': counter.completion_tokens,
        'total': counter.total_tokens
    }

    return res_state, elapsed, tokens_dict

In [6]:
q_bank = read_tasks("./../queries/bio_queries.yaml")
md_maker = MDGenerationFlow(read_prompts('./../prompts/md_generation/m2_dag_reasoning_prompt.txt'), MODEL, MODEL_PROVIDER, TEMPERATURE)
md, el_t, tokens = generate_meta(md_maker, q_bank[0])

In [7]:
'md_schema' in md['generation'].keys()

True

In [9]:
md['generation']

{'short_name': 'SHP-001',
 'description': 'Unique short code for the patient within the study.',
 'md_schema': {'type': {'keyword': 'categorical',
   'description': 'Short alphanumeric identifier unique per patient.',
   'type': 'string',
   'md_field_class': 'study_object',
   'reason_of_class': 'Identifies the object of study (patient) rather than a technical artifact.',
   'dag_step': 'patient_enrollment',
   'dag_descr': 'Patient consented and enrolled in the study.',
   'dag_childs': 'biopsy_acquisition'}}}

## 1. Generation a lot of metadatas (for averaging results) and thier embeddings

In [12]:
import pandas as pd
import os
import time


QUERIES_BANKS = {
    "synth_bio":'./../queries/bio_queries.yaml',
}

MD_GEN_PROMPTS = {
    'basic': './../prompts/md_generation/m2_dag_reasoning_prompt.txt',
}

RUNS_CNT = 8
DEST_DICT = './../data/openai-oss/md_generations/m2'
bank_dicts = []
os.makedirs(DEST_DICT, exist_ok=True)

In [13]:
# iterate over q_banks...
for q_name, q_path in QUERIES_BANKS.items():
    q_bank = read_tasks(q_path)
    
    # ...now over queries...
    for q_num, q in enumerate(q_bank):
        # ...also iterate over prompts of generation...
        for pr_name, pr_path in MD_GEN_PROMPTS.items():
            md_maker = MDGenerationFlow(read_prompts(pr_path), MODEL, MODEL_PROVIDER, TEMPERATURE)

            # ...and now over runs for averaging
            for run in range(RUNS_CNT):
                # create md!
                is_complete = False
                while not is_complete:
                    try:
                        tmp_dicts = []
                        md, el_t, tokens = generate_meta(md_maker, q)
                        md = md['generation']

                        for k, v in md['md_schema'].items():
                            new_dict = {
                                'query_id': q_num,
                                'query_topic': q['topic'],
                                'query_field_of_science': q['field_of_science'],
                                'query_llm_query': q['llm_query'],
                                'run_id': run,
                                'md_short_name': md['short_name'],
                                'md_description': md['description'],
                                'field_name': k,
                                'field_description': v['descr'],
                                'field_type': v['type'],
                                'field_class': v['md_field_class'],
                                'field_class_reason': v['reason_of_class'],
                                'dag_step': v['dag_step'],
                                'dag_descr': v['dag_descr'],
                                'dag_childs': v['dag_childs'],
                                'elapsed_time': el_t,
                                'tokens': str(tokens)
                            }
                            tmp_dicts.append(new_dict)
                        
                        if len(tmp_dicts) == 0:
                            print(f'Empty dicts - rework (wasted: {el_t, tokens}).')
                        else:
                            is_complete = True

                    except Exception as e:
                        print(f'Exception {str(e)} catching. Retrying...')
                        time.sleep(120)
                        continue
                
                bank_dicts += tmp_dicts
                print(f'Finish processing: {os.path.join(q_name, str(q_num), pr_name, f"r_{run+1}")}', 'elapsed_time', el_t, 'tokens', str(tokens))

    df = pd.DataFrame(bank_dicts)
    df.to_csv(os.path.join(DEST_DICT, f'm2_{q_name}_mds.csv'))


Finish processing: synth_bio/0/basic/r_1 elapsed_time 60.766435437995824 tokens {'prompt': 1198, 'completion': 11084, 'total': 12282}
Exception 'descr' catching. Retrying...
Finish processing: synth_bio/0/basic/r_2 elapsed_time 57.99451665500237 tokens {'prompt': 1198, 'completion': 8993, 'total': 10191}
Finish processing: synth_bio/0/basic/r_3 elapsed_time 57.13886131699837 tokens {'prompt': 1198, 'completion': 7327, 'total': 8525}
Finish processing: synth_bio/0/basic/r_4 elapsed_time 62.8652333469945 tokens {'prompt': 1198, 'completion': 8562, 'total': 9760}
Exception 'reason_of_class' catching. Retrying...
Exception 'descr' catching. Retrying...
Finish processing: synth_bio/0/basic/r_5 elapsed_time 56.93593435200455 tokens {'prompt': 1198, 'completion': 6514, 'total': 7712}
Finish processing: synth_bio/0/basic/r_6 elapsed_time 77.51362609500211 tokens {'prompt': 1198, 'completion': 7759, 'total': 8957}
Finish processing: synth_bio/0/basic/r_7 elapsed_time 48.08950837199518 tokens {'

In [ ]:
os.makedirs(DEST_DICT, exist_ok=True)
df.to_csv(os.path.join(DEST_DICT, f'm2_{q_name}_mds.csv'), escapechar='\\')

In [ ]:
tmp_df = pd.DataFrame(bank_dicts)
tmp_df.to_csv(os.path.join(DEST_DICT, f'm2_{q_name}_mds.csv'), escapechar='\\')


In [ ]:
import pickle

with open('m2_dict_pickle.pkl', 'wb') as fio:
    pickle.dump(bank_dicts, fio)

In [ ]:
import os
import pandas as pd
from langchain_openai import OpenAIEmbeddings
from eval.md_classes import *

QUERIES_CNT = 16
RUNS = 8

df = pd.read_csv('./../data/qwen/md_generations/m2_synth_bio_mds.csv', index_col=[0])
openai_embedder = OpenAIEmbeddings(model="text-embedding-3-large")


In [ ]:
row = df.iloc[0]
if 'dag_step' in row.keys():
    block = DataStep(row['dag_step'], row['dag_descr'])
else:
    block = None
md_field = MDField(row['field_name'], row['field_description'], row['field_type'], block=block)
fields_str = str(md_field)
fields_str

In [ ]:
from eval.md_classes import DataStep, MDField

In [ ]:
os.makedirs('./embeds/m2_fields_only/')

for idx, row in df.iterrows():
    #if 'dag_step' in row.keys():
    #    block = DataStep(row['dag_step'], row['dag_descr'])
    #else:
    block = None
    md_field = MDField(row['field_name'], row['field_description'], row['field_type'], block=block)
    fields_str = str(md_field)

    emb = openai_embedder.embed_query(fields_str)
    emb = np.array(emb)
    np.save(os.path.join('./embeds/m2_fields_only/', f'{idx}.npy'), emb)
    print(f"(shape {emb.shape}) - '{fields_str}' proceded.")